# Generation run (Colab)
Use an **L4 or A100** runtime (a T4 works but sampling is ~20x slower). Run cells 1-3 at the start of every session, then continue with the first unfinished step. Every step is resumable: re-running it skips finished questions.

**To stop a running cell, use the stop button** (not Disconnect) so the current file is closed and saved to Drive.

In [3]:
# 1. Mount Drive and get/update the code (outputs live in Drive next to it)
from google.colab import drive; drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/coms4705-hallucination-gen'
REPO_URL = 'https://github.com/nataliecclaire/NLP_final_project.git'
if not os.path.exists(PROJ):
    !git clone {REPO_URL} {PROJ}
%cd {PROJ}
!git pull --ff-only
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
/content/drive/MyDrive/coms4705-hallucination-gen
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 4.06 KiB | 6.00 KiB/s, done.
From https://github.com/nataliecclaire/NLP_final_project
   2013ce5..e66512b  main       -> origin/main
Updating f4c7169..e66512b
error: The following untracked working tree files would be overwritten by merge:
	outputs/generations/triviaqa_train.jsonl
Please move or remove them before you merge.
Aborting
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [4]:
# 2. Install
!pip install -q -r requirements.txt

In [5]:
# 3. (Optional) Hugging Face login with a Read token - the model is not gated; this just avoids download rate limits
from huggingface_hub import login; login()

## Step A: 50 TriviaQA questions end to end (done)

In [ ]:
!python scripts/generate.py --dataset triviaqa --split train --limit 50

In [ ]:
!python -m pytest -s

## Step B: full run
### B1. Greedy answers (HF transformers, one question at a time)
GSM8K is the slow part (~20 s/question on an L4, ~3 h for all 500).

In [6]:
!python scripts/generate.py --dataset all --split all --phase greedy

config.json: 100% 596/596 [00:00<00:00, 2.79MB/s]
tokenizer_config.json: 100% 2.10k/2.10k [00:00<00:00, 1.24MB/s]
tokenizer.json: 100% 1.80M/1.80M [00:00<00:00, 23.8MB/s]

tokenizer.model: downloading bytes:   0% 0.00/493k [00:00<?, ?B/s]
tokenizer.model: downloading bytes: 100% 339k/339k [00:01<00:00, 281kB/s, 32.9kB/s  ]
tokenizer.model: reconstructing file: 100% 493k/493k [00:01<00:00, 409kB/s, 47.9kB/s  ]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.33MB/s]
model.safetensors.index.json: 100% 25.1k/25.1k [00:00<00:00, 55.7MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.54G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/14.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   6% 939M/14.5G [00:10<03:56, 57.3MB/s,  219MB/s  ]
Reconstructing (incomplete total...):  12% 1.68G/14.5G [00:29<05:01, 42.5MB/s, 51.7MB/s  ]


### B2. Install vLLM for fast sampling
If this fails, skip to B4: it falls back to HF transformers (much slower).

In [ ]:
!pip install -q vllm
!python -c "import vllm; print('vLLM', vllm.__version__)"

### B3. Check vLLM against the saved HF greedy outputs
Must end with `vLLM CHECK PASSED` before running B4 with vLLM.

In [ ]:
!python scripts/check_vllm.py

### B4. 10 samples per question

In [ ]:
!python scripts/generate.py --dataset all --split all --phase sample

### B5. Merge into the final files + summary

In [ ]:
!python scripts/generate.py --dataset all --split all --phase merge

### B6. Make sure everything is written to Drive before closing the session

In [ ]:
from google.colab import drive; drive.flush_and_unmount()